# Notebook 13: Ensemble Strategy - Final Evaluation

## Objetivo
Combinar DeBERTa-v3-large y Llama-3.1-8B mediante soft voting ensemble.

## CRÍTICO: Patrón de Inferencia Secuencial
Para evitar errores CUDA OOM, NUNCA cargamos ambos modelos simultáneamente.

## 1. Imports

In [ ]:
# Imports: bibliotecas para deep learning, metricas y manejo de modelos transformer
import torch
import pandas as pd
import numpy as np
import gc
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import time
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,  # Configuracion para cuantizacion 4-bit (QLoRA)
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score
)
from peft import PeftModel  # Para cargar adaptadores LoRA/QLoRA

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 2. Rutas del Proyecto

In [ ]:
# Configuracion de rutas del proyecto y directorios clave
def find_root() -> Path:
    current = Path.cwd()
    for cand in [current, *current.parents]:
        if (cand / "data" / "raw").exists():
            return cand
    raise FileNotFoundError("No se encontro data/raw")

PROJECT_ROOT = find_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
BOUNDARIES_DIR = PROJECT_ROOT / "data" / "processed" / "boundaries"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "finetuning"  # Modelos fine-tuneados
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ensemble"
REPORTS_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 256  # Longitud maxima de secuencia para tokenizacion

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 3. Funciones de Carga

In [ ]:
# Funciones para cargar oraciones desde archivos y crear pares de texto
def load_sentences(level: str, split: str, doc_id: str) -> list[str]:
    """Carga oraciones de un documento divididas por lineas."""
    path = DATA_RAW / level / split / f"{doc_id}.txt"
    if not path.exists():
        raise FileNotFoundError(f"No encontrado: {path}")
    text = path.read_text(encoding='utf-8').strip()
    sentences = [s.strip() for s in text.split('\n') if s.strip()]
    return sentences

def create_text_pairs(df: pd.DataFrame, desc: str = "Procesando") -> list[dict]:
    """Crea pares de oraciones consecutivas para deteccion de fronteras de autor."""
    pairs = []
    skipped = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            sentences = load_sentences(row['level'], row['split'], row['doc_id'])
            # Verificar indices validos
            if row['sent_left_id'] >= len(sentences) or row['sent_right_id'] >= len(sentences):
                skipped += 1
                continue
            text_a = sentences[row['sent_left_id']]
            text_b = sentences[row['sent_right_id']]
            if not text_a or not text_b:
                skipped += 1
                continue
            pairs.append({
                'text_a': text_a,
                'text_b': text_b,
                'label': int(row['y']),  # 0: mismo autor, 1: frontera entre autores
                'level': row['level'],
                'doc_id': row['doc_id']
            })
        except Exception as e:
            skipped += 1
    if skipped > 0:
        print(f"Saltados: {skipped}")
    return pairs

print("Funciones OK")

## 4. Cargar Datos

In [ ]:
# Cargar datos de validacion para evaluacion del ensemble
boundaries_val = pd.read_csv(BOUNDARIES_DIR / "boundaries_validation.csv")
print(f"Boundaries: {len(boundaries_val):,}")
print(boundaries_val['y'].value_counts())

# Crear pares de texto con sus etiquetas
val_pairs = create_text_pairs(boundaries_val)
df_val = pd.DataFrame(val_pairs)
print(f"\nDataset: {len(df_val):,} ejemplos")

## 5. Config Modelos

In [ ]:
# Configuracion de los dos modelos a combinar en el ensemble
# DeBERTa: modelo encoder-only fine-tuneado con entrenamiento estandar
# Llama: modelo decoder-only fine-tuneado con QLoRA (4-bit quantization) y prompts instructivos
MODEL_CONFIGS = {
    'deberta': {
        'checkpoint': CHECKPOINTS_DIR / 'deberta-v3-large' / 'best_model',
        'base_model': 'microsoft/deberta-v3-large',
        'batch_size': 64,
        'use_qlora': False,  # Entrenamiento completo (no cuantizado)
        'use_instructive': False  # Input directo: [text_a, text_b]
    },
    'llama': {
        'checkpoint': CHECKPOINTS_DIR / 'llama-3.1-8b' / 'best_model',
        'base_model': 'meta-llama/Meta-Llama-3.1-8B-Instruct',
        'batch_size': 32,
        'use_qlora': True,  # QLoRA: adaptadores de bajo rango + cuantizacion 4-bit
        'use_instructive': True  # Input con prompt: "Analyze the writing style..."
    }
}

for name, cfg in MODEL_CONFIGS.items():
    ok = "OK" if cfg['checkpoint'].exists() else "X"
    print(f"{ok} {name}")

## 6. Función de Inferencia Secuencial ROBUSTA

In [ ]:
def get_probas_sequentially(model_key, df_val):
    """
    Inferencia secuencial para evitar CUDA OOM: cargar modelo -> predecir -> limpiar memoria.
    
    Retorna probabilidades de clasificacion (shape: [N, 2]) donde:
    - probas[:, 0]: P(clase=0) = misma autoria
    - probas[:, 1]: P(clase=1) = frontera entre autores
    
    FLUJO:
    1. Cargar tokenizer desde checkpoint (critico para vocabulario de Llama)
    2. Tokenizar dataset (con o sin prompt instructivo segun modelo)
    3. Cargar modelo base + adaptador PEFT si usa QLoRA
    4. Ejecutar prediccion via Trainer
    5. Convertir logits a probabilidades con softmax
    6. Limpiar memoria GPU/RAM antes de retornar
    
    FIXES CRITICOS:
    - Ajustar pad_token_id en model.config para batch_size > 1
    - Resolver mismatch de vocabulario en Llama (128257 vs 128256)
    - Redimensionar embeddings ANTES de cargar adaptador PEFT
    - NO hacer merge de PEFT (usar adaptador directamente)
    """
    cfg = MODEL_CONFIGS[model_key]
    print(f"{'='*70}{model_key.upper()}{'='*70}")
    
    # Pre-limpieza de memoria
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    # 1. TOKENIZER: cargar desde checkpoint para mantener vocabulario correcto
    print(f"[1/5] Loading tokenizer from {cfg['checkpoint']}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(cfg['checkpoint']))
        print(f"  OK Loaded from checkpoint (vocab={len(tokenizer)})")
    except Exception as e:
        print(f"  WARNING Fallback to base model")
        tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
    
    # Configurar pad_token si no existe (necesario para batching)
    if tokenizer.pad_token is None:
        if cfg['use_qlora']:  # Llama: crear token especial
            tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
            print(f"  OK Created pad_token '<|pad|>' (ID: {tokenizer.pad_token_id})")
        else:  # DeBERTa: reutilizar eos_token
            tokenizer.pad_token = tokenizer.eos_token
            print(f"  OK Set pad_token = eos_token")
    print(f"  Tokenizer vocab: {len(tokenizer)}, pad_token_id: {tokenizer.pad_token_id}")
    
    # 2. TOKENIZACION: con o sin prompt instructivo
    print(f"[2/5] Tokenizing...")
    dataset = Dataset.from_pandas(df_val[['text_a', 'text_b', 'label']])
    
    if cfg['use_instructive']:
        # Llama: formato instructivo con descripcion de tarea
            def tokenize_fn(ex):
                texts = [
                    f"Analyze the writing style of these two consecutive sentences and determine if they were written by the same author.\n\n"
                    f"Text A: {a}\n"
                    f"Text B: {b}\n\n"
                    f"Are these sentences from the same author?"
                    for a, b in zip(ex['text_a'], ex['text_b'])
                ]
                return tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=False)
    else:
        # DeBERTa: concatenacion estandar [CLS] text_a [SEP] text_b [SEP]
        def tokenize_fn(ex):
            return tokenizer(ex['text_a'], ex['text_b'], truncation=True, 
                           max_length=MAX_LENGTH, padding=False)
    
    tok_ds = dataset.map(tokenize_fn, batched=True)
    tok_ds = tok_ds.rename_column('label', 'labels')
    tok_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    print(f"  OK Tokenized: {len(tok_ds)}")
    
    # 3. CARGA DE MODELO: path diferente para QLoRA vs estandar
    print(f"[3/5] Loading model...")
    if cfg['use_qlora']:
        # Path QLoRA (Llama): base model cuantizado + adaptador PEFT
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,  # Cuantizacion 4-bit para reducir memoria
            bnb_4bit_use_double_quant=True,  # Cuantizacion doble para mayor precision
            bnb_4bit_quant_type="nf4",  # NormalFloat4: cuantizacion optimizada
            bnb_4bit_compute_dtype=torch.bfloat16  # Computaciones en bfloat16
        )
        base = AutoModelForSequenceClassification.from_pretrained(
            cfg['base_model'], 
            num_labels=2,
            quantization_config=bnb_cfg,
            device_map='auto',  # Distribucion automatica en GPUs disponibles
            trust_remote_code=True
        )
        print(f"  Base model vocab: {base.config.vocab_size}")
        print(f"  Tokenizer vocab: {len(tokenizer)}")
        
        # FIX: Resolver mismatch de vocabulario (Llama añade pad_token)
        if len(tokenizer) != base.config.vocab_size:
            print(f"  WARNING Vocabulary mismatch detected!")
            print(f"    Model: {base.config.vocab_size} tokens")
            print(f"    Tokenizer: {len(tokenizer)} tokens")
            print(f"  -> Resizing embeddings to {len(tokenizer)}...")
            base.resize_token_embeddings(len(tokenizer))
            print(f"  OK Embeddings resized")
        else:
            print(f"  OK Vocab sizes match")
        
        # FIX: Configurar pad_token_id ANTES de cargar PEFT
        base.config.pad_token_id = tokenizer.pad_token_id
        print(f"  OK Set base.config.pad_token_id = {tokenizer.pad_token_id}")
        
        # Cargar adaptador PEFT (NO hacer merge para mantener precision)
        print(f"  Loading PEFT adapter...")
        model = PeftModel.from_pretrained(base, str(cfg['checkpoint']))
        print(f"  OK PEFT adapter loaded (keeping as adapter, NOT merging)")
        
    else:
        # Path estandar (DeBERTa): modelo completo fine-tuneado
        model = AutoModelForSequenceClassification.from_pretrained(
            str(cfg['checkpoint']), 
            num_labels=2
        )
        
        # FIX: Configurar pad_token_id tambien para modelos estandar
        model.config.pad_token_id = tokenizer.pad_token_id
        print(f"  OK Set model.config.pad_token_id = {tokenizer.pad_token_id}")
        
        if torch.cuda.is_available():
            model = model.to('cuda')
    
    model.eval()
    print(f"  OK Model ready for inference")
    
    # 4. TRAINER: configurar para inferencia con precision mixta
    print(f"[4/5] Setting up trainer (batch={cfg['batch_size']})...")
    args = TrainingArguments(
        output_dir='./tmp',
        per_device_eval_batch_size=cfg['batch_size'],
        fp16=not cfg['use_qlora'],  # FP16 para DeBERTa
        bf16=cfg['use_qlora'],  # BFloat16 para QLoRA
        dataloader_num_workers=4,
        remove_unused_columns=False
    )
    # Dynamic padding: ajustar padding al batch actual (mas eficiente)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    trainer = Trainer(
        model=model, 
        args=args, 
        tokenizer=tokenizer,
        data_collator=data_collator
    )
    
    # 5. PREDICCION: obtener logits y convertir a probabilidades
    print(f"[5/5] Predicting...")
    preds = trainer.predict(tok_ds)
    logits = preds.predictions  # Shape: [N, 2] (logits sin normalizar)
    # Softmax: convertir logits a probabilidades [0, 1] que sumen 1
    probas = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()
    print(f"  OK Probas shape: {probas.shape}")
    print(f"  Sample (first 3):")
    for i in range(min(3, len(probas))):
        print(f"    [{i}] P(class=0)={probas[i,0]:.4f}, P(class=1)={probas[i,1]:.4f}")
    
    # 6. LIMPIEZA: liberar memoria para el siguiente modelo
    print(f"{'='*70}")
    print(f"CLEANUP: Releasing memory...")
    print(f"{'='*70}")
    del model, trainer, tokenizer, tok_ds, dataset
    if cfg['use_qlora']:
        del base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"OK {model_key.upper()} INFERENCE COMPLETE")
    
    return probas

print("Inference function ready")

## 7. Ejecutar Inferencia Secuencial

In [ ]:
# Ejecutar inferencia secuencial: DeBERTa primero (mas rapido)
# Patron secuencial: cargar -> predecir -> limpiar -> siguiente modelo
# Evita CUDA OOM al nunca tener ambos modelos en memoria simultaneamente
probs_deberta = get_probas_sequentially('deberta', df_val)
time.sleep(3)  # Pausa para asegurar limpieza completa de GPU

In [ ]:
# Llama segundo (modelo mas grande, requiere mas memoria)
probs_llama = get_probas_sequentially('llama', df_val)

## 8. Verificación

In [ ]:
# Verificar dimensiones de arrays de probabilidades
print(f"DeBERTa: {probs_deberta.shape}")
print(f"Llama: {probs_llama.shape}")

# Guardar probabilidades para analisis posterior
np.save(OUTPUT_DIR / 'probs_deberta_val.npy', probs_deberta)
np.save(OUTPUT_DIR / 'probs_llama_val.npy', probs_llama)
print(f"OK Saved to {OUTPUT_DIR}")

## 9. Optimización de Ensemble

In [ ]:
# Optimizacion de alpha mediante grid search exhaustivo
# SOFT VOTING: combinar probabilidades de ambos modelos con peso alpha
# Formula: ensemble_probs = alpha * deberta_probs + (1-alpha) * llama_probs
# - alpha=1.0: solo DeBERTa
# - alpha=0.0: solo Llama
# - alpha=0.5: promedio equiponderado
# Objetivo: encontrar alpha que maximiza F1-macro en validacion

y_true = df_val['label'].values

alphas = np.arange(0, 1.01, 0.01)  # Grid de 101 valores [0.00, 0.01, ..., 1.00]
f1_scores = []

print("Buscando mejor alpha...")
for alpha in alphas:
    # Combinacion lineal de probabilidades (soft voting)
    probs = alpha * probs_deberta + (1 - alpha) * probs_llama
    # Clasificacion: argmax de probabilidades combinadas
    y_pred = np.argmax(probs, axis=1)
    # Calcular F1-macro (promedio de F1 para cada clase)
    f1 = f1_score(y_true, y_pred, average='macro')
    f1_scores.append(f1)

f1_scores = np.array(f1_scores)
best_idx = np.argmax(f1_scores)
best_alpha = alphas[best_idx]
best_f1 = f1_scores[best_idx]

print(f"\nBest alpha: {best_alpha:.2f}")
print(f"Best F1: {best_f1:.4f}")
print(f"Composition: {best_alpha*100:.0f}% DeBERTa + {(1-best_alpha)*100:.0f}% Llama")

## 10. Evaluación Final

In [ ]:
# Evaluacion final del ensemble con alpha optimo
# Aplicar soft voting con el alpha que maximizo F1 en grid search
probs_ensemble = best_alpha * probs_deberta + (1 - best_alpha) * probs_llama
y_pred_ensemble = np.argmax(probs_ensemble, axis=1)

print("="*70)
print("ENSEMBLE FINAL")
print("="*70)
print(classification_report(y_true, y_pred_ensemble, 
                           target_names=['Non-Boundary', 'Boundary'], digits=4))

# Matriz de confusion para analizar errores
cm = confusion_matrix(y_true, y_pred_ensemble)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Non-B', 'Boundary'],
           yticklabels=['Non-B', 'Boundary'])
plt.title(f'Ensemble (alpha={best_alpha:.2f})')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=300)
plt.show()

## 11. Comparación Global

In [ ]:
# Comparacion de rendimiento: todos los modelos del proyecto
historic = []

# Baseline (Notebook 09): RandomForest con TF-IDF
try:
    with open(REPORTS_DIR / '09_metrics.json') as f:
        data = json.load(f)
    historic.append({'Model': 'Baseline', 'F1': data.get('f1_macro', 0.707)})
except:
    historic.append({'Model': 'Baseline', 'F1': 0.707})

# ICL (Notebook 11): In-Context Learning sin fine-tuning
try:
    with open(REPORTS_DIR / '11_icl_metrics_final.json') as f:
        data = json.load(f)
    historic.append({'Model': 'ICL', 'F1': data.get('f1_macro', 0.383)})
except:
    historic.append({'Model': 'ICL', 'F1': 0.383})

# DeBERTa fine-tuned (Notebook 12a)
y_pred_deb = np.argmax(probs_deberta, axis=1)
historic.append({'Model': 'DeBERTa-FT', 'F1': f1_score(y_true, y_pred_deb, average='macro')})

# Llama fine-tuned (Notebook 12b)
y_pred_llm = np.argmax(probs_llama, axis=1)
historic.append({'Model': 'Llama-FT', 'F1': f1_score(y_true, y_pred_llm, average='macro')})

# Ensemble (Notebook 13): soft voting con alpha optimo
historic.append({'Model': f'Ensemble (alpha={best_alpha:.2f})', 'F1': best_f1})

df_hist = pd.DataFrame(historic)
print("\nEVOLUCION DEL PROYECTO")
print(df_hist.to_string(index=False))

# Grafico comparativo de rendimiento
plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(df_hist)), df_hist['F1'], 
              color=['gray', 'red', 'blue', 'purple', 'green'])
plt.xticks(range(len(df_hist)), df_hist['Model'], rotation=45, ha='right')
plt.ylabel('F1 Macro')
plt.title('Project Evolution')
plt.grid(axis='y', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, df_hist['F1'])):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'global_comparison.png', dpi=300)
plt.show()

# Calcular mejora total respecto al baseline
improvement = ((best_f1 - historic[0]['F1']) / historic[0]['F1']) * 100
print(f"\nOK Mejora total: {improvement:+.2f}%")

## 12. Guardar Reporte Final

In [ ]:
# Guardar reporte JSON con metricas finales del ensemble
report = {
    'ensemble': {
        'alpha': float(best_alpha),  # Peso optimo de DeBERTa en soft voting
        'f1_macro': float(best_f1),
        'accuracy': float(accuracy_score(y_true, y_pred_ensemble))
    },
    'individual': {
        'deberta_f1': float(f1_score(y_true, y_pred_deb, average='macro')),
        'llama_f1': float(f1_score(y_true, y_pred_llm, average='macro'))
    },
    'improvement': float(improvement)  # Porcentaje de mejora respecto al baseline
}

with open(OUTPUT_DIR / 'ensemble_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nOK Report saved: {OUTPUT_DIR / 'ensemble_report.json'}")
print(f"\n{'='*70}")
print("EVALUACION COMPLETA")
print(f"{'='*70}")
print(f"Best alpha: {best_alpha:.2f}")
print(f"Ensemble F1: {best_f1:.4f}")
print(f"Improvement: {improvement:+.2f}%")
print(f"{'='*70}")